# NeoBERT
"NeoBERT: A Next-Generation BERT" (Le Breton et al., 2025)

**Idea:** BERT's architecture is from 2018; decoders kept improving (RoPE, RMSNorm, SwiGLU, pre-norm) and encoders mostly did not. NeoBERT is a BERT rebuilt with the modern decoder block, a deeper-and-narrower shape, a simpler and heavier masking recipe, and far more data. Same job as BERT: a bidirectional encoder pretrained with masked language modeling, then fine-tuned.

### BERT vs NeoBERT
| | BERT-base (`bert.ipynb`) | NeoBERT |
|---|---|---|
| Position | learned absolute, added to the input | **RoPE** on $q, k$ in every layer (`rope.ipynb`) |
| Norm | Post-LN, LayerNorm | **Pre-norm, RMSNorm** |
| FFN | GELU, $4D$ | **SwiGLU**, $\frac{8}{3}D$ |
| Input embeddings | token + position + segment | **token only** |
| Depth x width | 12 x 768 (110M) | **28 x 768** (250M): deeper, same width |
| Max length | 512 | **4096** (1M steps at 1024, then 50k steps at 4096) |
| Pretraining tasks | MLM + next-sentence prediction | **MLM only** |
| Masking | 15%, 80/10/10 ([MASK] / random / unchanged) | **20%, always [MASK]** |
| Schedule | AdamW, linear warmup + linear decay | AdamW, linear warmup + **cosine decay** |
| Data | BooksCorpus + Wikipedia (3.3B words) | **RefinedWeb (600B tokens)** |
| Tokenizer | WordPiece 30k | WordPiece 30k (unchanged) |

`bert.ipynb` explains MLM, the `[CLS]` head and fine-tuning in full. This notebook reuses its data pipeline and builds only what NeoBERT changes: the encoder block, the embeddings and the masking / schedule.

Shape flow: `ids (B, T) -> token embed (B, T, D) -> [RMSNorm -> attn(RoPE, pad mask) -> + -> RMSNorm -> SwiGLU -> +] x L -> RMSNorm -> (B, T, D) -> MLM logits at masked positions (M, V) | [CLS] (B, D) -> logits (B, 2)`

In [ ]:
import re
import copy
import math
from collections import Counter

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - Enron spam emails (same pipeline as `bert.ipynb`)
Word-level vocab of 10k, `[CLS] subject [SEP] body [SEP] [PAD]...`, $T = 128$, 80/20 split. The one difference: **no segment ids**. NeoBERT drops next-sentence prediction and with it the sentence A / B embedding; `[SEP]` alone marks the boundary.

In [ ]:
# the original Enron-Spam csv (github.com/MWiechmann/enron_spam_data); in the older local enron_spam_data.csv the spam rows are 6 ham emails repeated
df = pd.read_csv("../datasets/enron_spam_data_original.csv")
df["Subject"] = df["Subject"].fillna("")
df["Message"] = df["Message"].fillna("")
df["label"] = (df["Spam/Ham"] == "spam").astype(int)   # 1 = spam, 0 = ham
df = df.sample(frac=1.0, random_state=0).reset_index(drop=True)
n_train = int(0.8 * len(df))
train_df, test_df = df[:n_train], df[n_train:]

PAD, CLS, SEP, MASK, UNK = 0, 1, 2, 3, 4
specials = ["[PAD]", "[CLS]", "[SEP]", "[MASK]", "[UNK]"]
vocab_size, max_len, max_subject = 10000, 128, 16

def tokenize(text):
    return re.findall(r"\[mask\]|[a-z0-9]+|[^\sa-z0-9]", text.lower()[:3000])

counts = Counter(w for t in train_df["Subject"] + " " + train_df["Message"] for w in tokenize(t))
itos = specials + [w for w, _ in counts.most_common(vocab_size - len(specials))]
stoi = {w: i for i, w in enumerate(itos)}

def encode(subject, message):
    # -> ids (T,): [CLS] subject [SEP] message [SEP] [PAD]...
    words = lambda text: [MASK if w == "[mask]" else stoi.get(w, UNK) for w in tokenize(text)]
    a = words(subject)[:max_subject]
    b = words(message)[:max_len - len(a) - 3]
    ids = [CLS] + a + [SEP] + b + [SEP]
    return ids + [PAD] * (max_len - len(ids))

def encode_df(d):
    return torch.tensor([encode(s, m) for s, m in zip(d["Subject"], d["Message"])]), torch.tensor(d["label"].values)

train_ids, train_y = encode_df(train_df)     # (N, T), (N,)
test_ids, test_y = encode_df(test_df)
print("vocab:", len(itos), "| train:", tuple(train_ids.shape), "| test:", tuple(test_ids.shape), "| spam fraction:", round(train_y.float().mean().item(), 3))

## 2. The NeoBERT block
The Llama decoder block from `slm.ipynb` with one change: the causal mask is replaced by a **padding mask**, so attention is bidirectional. RMSNorm, RoPE and SwiGLU formulas are in `slm.ipynb` / `rope.ipynb`.

### Formula: Pre-norm encoder block
$$h' = h + \text{Attn}(\text{RMSNorm}(h)), \qquad h_{out} = h' + \text{SwiGLU}(\text{RMSNorm}(h'))$$
- BERT: $h' = \text{LN}(h + \text{Attn}(h))$ (Post-LN). Pre-norm keeps the residual path clean, which is what lets NeoBERT go to 28 layers

### Formula: Bidirectional attention with RoPE
$$\text{score}_{mn} = \frac{(R_m q_m)^T (R_n k_n)}{\sqrt{d}} + M_n, \qquad M_n = \begin{cases} 0 & x_n \ne \texttt{[PAD]} \\ -\infty & x_n = \texttt{[PAD]} \end{cases}$$
- $R_m$: RoPE rotation for position $m$; the score depends on the offset $m - n$, which here can be **negative or positive** (tokens to the right are visible)
- position is injected in every layer through $q, k$, so the input embedding is the token embedding alone: no position table, no segment table, and no hard maximum length
- this is what allows the two-stage training (1024, then 4096 tokens) without resizing anything

### Formula: Depth-to-width
$$\text{params} \approx L \cdot \big(4D^2 + 3 \cdot D \cdot \tfrac{8}{3}D\big) = 12\,L\,D^2$$
- attention has $4D^2$ weights, SwiGLU $3 \cdot D \cdot H$ with $H = \frac{8}{3}D$
- BERT-large widens ($D = 1024$, $L = 24$); NeoBERT keeps $D = 768$ and deepens to $L = 28$. The paper's argument: at $D = 768$, 12 layers are too shallow for the width ("width-inefficient"), and keeping 768 makes it a drop-in replacement for base models

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def rope_cache(T, head_dim, base=10000):
    freqs = 1.0 / base ** (torch.arange(0, head_dim, 2).float() / head_dim)
    angles = torch.arange(T).float()[:, None] * freqs[None]
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


class Attention(nn.Module):
    # (B, T, D), pad_mask (B, T) -> (B, T, D); bidirectional
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads, self.head_dim = num_heads, dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, pad_mask, cos, sin):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)                                 # each (B, heads, T, hd)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5               # (B, heads, T, T)
        att = att.masked_fill(pad_mask[:, None, None, :], float("-inf")).softmax(dim=-1)   # only [PAD] keys are hidden
        return self.proj((att @ v).transpose(1, 2).reshape(B, T, D))


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class EncoderBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1, self.attn = RMSNorm(dim), Attention(dim, num_heads)
        self.norm2, self.ffn = RMSNorm(dim), SwiGLU(dim, 64 * ((8 * dim // 3 + 63) // 64))

    def forward(self, x, pad_mask, cos, sin):
        x = x + self.attn(self.norm1(x), pad_mask, cos, sin)
        return x + self.ffn(self.norm2(x))


class NeoBERT(nn.Module):
    # ids (B, T) -> hidden (B, T, D); heads: mlm_logits, classify
    def __init__(self, vocab_size, max_len=4096, dim=192, depth=8, num_heads=4, num_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim, padding_idx=PAD)       # the ONLY input embedding
        self.blocks = nn.ModuleList([EncoderBlock(dim, num_heads) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        self.mlm_transform = nn.Sequential(nn.Linear(dim, dim), nn.GELU(), RMSNorm(dim))
        self.mlm_bias = nn.Parameter(torch.zeros(vocab_size))
        self.classifier = nn.Sequential(nn.Linear(dim, dim), nn.Tanh(), nn.Linear(dim, num_classes))
        self.apply(lambda m: nn.init.normal_(m.weight, std=0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)
        cos, sin = rope_cache(max_len, dim // num_heads)                  # a buffer, not parameters
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def forward(self, ids):
        T = ids.shape[1]
        pad_mask = ids == PAD
        x = self.embed(ids)                                               # (B, T, D): no position, no segment
        for block in self.blocks:
            x = block(x, pad_mask, self.cos[:T], self.sin[:T])
        return self.norm(x)

    def mlm_logits(self, h):
        return self.mlm_transform(h) @ self.embed.weight.T + self.mlm_bias   # (M, D) -> (M, V), tied to the embedding

    def classify(self, ids):
        return self.classifier(self.forward(ids)[:, 0])                   # [CLS] -> (B, 2)

model = NeoBERT(vocab_size).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M (depth 8, width 192; bert.ipynb: depth 4, width 256)")
x = train_ids[:2].to(device)
print(model(x).shape, model.classify(x).shape)

# no position table: the same weights run on any length up to the RoPE buffer (BERT would need 1000 rows of position embeddings)
long = torch.randint(5, vocab_size, (1, 1000), device=device)
print("runs on 1000 tokens:", model(long).shape)

## 3. Masking: 20%, always `[MASK]`
BERT masks 15% of tokens and replaces them 80/10/10 (`bert.ipynb` §6). NeoBERT follows later findings (Wettig et al., 2023: larger models can use higher rates and do not need the 80/10/10 rule) and uses a plain scheme.

### Formula: NeoBERT masking
$$m_t \sim \text{Bernoulli}(0.20) \ \ (t \text{ a word position}), \qquad \tilde x_t = \begin{cases} \texttt{[MASK]} & m_t = 1 \\ x_t & m_t = 0 \end{cases}$$
- 20% of positions are targets: a third more training signal per sequence than 15%, with a slightly more corrupted context
- no random-word or keep-unchanged cases
- masks are redrawn for every batch (dynamic masking, as in RoBERTa)

In [ ]:
def mask_tokens(ids, p=0.20):
    # ids (B, T) -> masked input (B, T), labels (B, T) with -100 at non-target positions
    ids = ids.clone()
    labels = ids.clone()
    target = (torch.rand(ids.shape, device=ids.device) < p) & (ids > UNK)   # real words only
    labels[~target] = -100
    ids[target] = MASK                                                      # 100% [MASK]
    return ids, labels

x, lab = mask_tokens(train_ids[:256])
print("fraction of word positions masked:", round(((lab != -100).sum() / (train_ids[:256] > UNK).sum()).item(), 3))
print("input: ", " ".join(itos[i] for i in x[0][:24].tolist()))

## 4. MLM pretraining
Same loss as `bert.ipynb` (cross-entropy over the masked positions only). Optimizer as in the paper: AdamW with $\beta = (0.9, 0.95)$, weight decay 0.1, gradient clipping at 1.0, and warmup followed by cosine decay.

### Formula: Linear warmup + cosine decay
$$\eta_t = \begin{cases} \eta \cdot \dfrac{t}{T_w} & t < T_w \\[2mm] \eta \cdot \Big[r + (1 - r)\,\tfrac{1}{2}\Big(1 + \cos \dfrac{\pi (t - T_w)}{T_{total} - T_w}\Big)\Big] & t \ge T_w \end{cases}$$
- $\eta = 6 \times 10^{-4}$: peak learning rate (the paper's value); $T_w$: warmup steps (300 here, 2000 in the paper)
- $r = 0.1$: the decay ends at 10% of the peak instead of 0
- BERT's linear decay spends more of training at low learning rates; cosine stays high longer

In [ ]:
pretrain_steps = 3000
batch_size = 64
lr = 6e-4
warmup = 300
log_every = 250

def warmup_cosine(warmup, total, floor=0.1):
    return lambda t: (t + 1) / warmup if t < warmup else floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * (t - warmup) / max(1, total - warmup)))

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine(warmup, pretrain_steps))

def mlm_step(model, ids):
    # -> loss, number correct, number of targets
    x, labels = mask_tokens(ids)
    target = labels != -100                                # (B, T)
    logits = model.mlm_logits(model(x)[target])            # (M, V)
    loss = F.cross_entropy(logits, labels[target])
    return loss, (logits.argmax(-1) == labels[target]).sum().item(), target.sum().item()

@torch.no_grad()
def mlm_eval(model, n_batches=20):
    model.eval()
    g = torch.Generator().manual_seed(0)
    tot_loss = tot_correct = tot_n = 0
    for _ in range(n_batches):
        ix = torch.randint(len(test_ids), (batch_size,), generator=g)
        loss, c, n = mlm_step(model, test_ids[ix].to(device))
        tot_loss += loss.item() * n; tot_correct += c; tot_n += n
    model.train()
    return tot_loss / tot_n, tot_correct / tot_n

history = []
model.train()
for step in range(1, pretrain_steps + 1):
    ix = torch.randint(len(train_ids), (batch_size,))
    loss, _, _ = mlm_step(model, train_ids[ix].to(device))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    if step % log_every == 0:
        test_loss, test_acc = mlm_eval(model)
        history.append((step, loss.item(), test_loss))
        print(f"step {step:5d} | train mlm loss {loss.item():.3f} | test mlm loss {test_loss:.3f} | test masked acc {test_acc:.3f} | lr {scheduler.get_last_lr()[0]:.2e}")

steps, tr, te = zip(*history)
plt.plot(steps, tr, label="train"); plt.plot(steps, te, label="test")
plt.xlabel("step"); plt.ylabel("MLM loss"); plt.legend(); plt.show()

## 5. Fine-tuning: spam vs ham
Same protocol as `bert.ipynb` §9 so the two notebooks can be compared: 1000 labeled emails, 600 steps, the pretrained encoder against the same architecture from random init. (Run `bert.ipynb` on the same csv for a like-for-like comparison; the MLM losses are not comparable, since NeoBERT masks 20% with a different replacement rule.)

In [ ]:
n_labeled, ft_steps, ft_batch, ft_lr, ft_warmup, eval_every = 1000, 600, 32, 1e-4, 50, 100

g = torch.Generator().manual_seed(0)
lab_ix = torch.randperm(len(train_ids), generator=g)[:n_labeled]
ft_ids, ft_y = train_ids[lab_ix], train_y[lab_ix]

@torch.no_grad()
def cls_accuracy(model, ids, y, bs=256):
    model.eval()
    correct = sum((model.classify(ids[i:i + bs].to(device)).argmax(-1).cpu() == y[i:i + bs]).sum().item() for i in range(0, len(ids), bs))
    model.train()
    return correct / len(ids)

def finetune(model, name):
    opt = torch.optim.AdamW(model.parameters(), lr=ft_lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, warmup_cosine(ft_warmup, ft_steps))
    torch.manual_seed(0)                                     # same batches for both runs
    curve = []
    model.train()
    for step in range(1, ft_steps + 1):
        ix = torch.randint(n_labeled, (ft_batch,))
        loss = F.cross_entropy(model.classify(ft_ids[ix].to(device)), ft_y[ix].to(device))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        if step % eval_every == 0:
            curve.append((step, cls_accuracy(model, test_ids, test_y)))
            print(f"[{name}] step {step:4d} | loss {loss.item():.4f} | test acc {curve[-1][1]:.4f}")
    return curve

curve_pt = finetune(copy.deepcopy(model), "pretrained")
torch.manual_seed(42)
curve_sc = finetune(NeoBERT(vocab_size).to(device), "scratch")

plt.plot(*zip(*curve_pt), marker="o", label="MLM-pretrained")
plt.plot(*zip(*curve_sc), marker="o", label="random init")
plt.xlabel("fine-tuning step"); plt.ylabel("test accuracy"); plt.legend(); plt.show()

## Notes
- The paper's ablation adds the changes one at a time on GLUE. The largest gains come from **data** (RefinedWeb, +3.6) and **scale** (to 250M with the deeper shape, +2.9), not from the block; the 20% masking and the optimizer change are roughly neutral on GLUE and kept for other reasons. Two things tried and rejected: a Llama BPE tokenizer and sequence packing.
- Results: 89.0 on GLUE dev and state of the art on MTEB among encoders of its class under identical fine-tuning, ahead of BERT-large, RoBERTa-large, NomicBERT and ModernBERT, with 250M parameters.
- For embeddings (MTEB) the sentence vector is the **mean** of the final hidden states, trained contrastively; the `[CLS]` head here is the classic classification setup from `bert.ipynb`.
- **ModernBERT** (Warner et al., 2024) is the close relative: also RoPE + pre-norm + GeGLU, but with alternating local/global attention, 8192 context and a 30% masking rate. Both show that encoders benefit from the same block changes as decoders.
- Why encoders still matter: retrieval, reranking, classification and embeddings for RAG run on encoders that are 10-100x smaller and faster than an LLM (`models.md`, RAG section).
- Sizes here are tiny (8 x 192). The real model is 28 x 768 with 12 heads, SwiGLU hidden 2048, and runs up to 4096 tokens.